# SRGL Statistical Analysis and Evaluation

This notebook performs comprehensive statistical analysis of SRGL system performance.

## Overview
- Load synthetic datasets and run SRGL predictions
- Calculate safety metrics (sensitivity, specificity, FNR, unsafe discharge rate)
- Perform statistical tests (McNemar's test, chi-square)
- Evaluate calibration (ECE, reliability diagrams)
- Compare with baseline methods

## Requirements
Run notebook 01 first to generate the datasets.

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict

from src.data_generator import PatientCase, RiskTier
from src.srgl import SRGL
from src.evaluation import SafetyMetrics, StatisticalTests, CalibrationMetrics

# Set random seed
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-paper')
sns.set_palette('colorblind')
%matplotlib inline

## 1. Load Datasets

In [ ]:
def load_cases(filepath: str) -> List[PatientCase]:
    """Load patient cases from JSON file."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    cases = []
    for case_data in data['cases']:
        # Reconstruct PatientCase objects
        case = PatientCase(
            case_id=case_data['case_id'],
            diagnosis=case_data['diagnosis'],
            ground_truth_tier=RiskTier(case_data['ground_truth_tier']),
            patient_features=case_data['patient_features'],
            clinical_presentation=type('obj', (object,), case_data['clinical_presentation'])(),
            vital_signs=case_data.get('vital_signs', {}),
            lab_results=case_data.get('lab_results', {}),
            timestamp=case_data.get('timestamp', '')
        )
        cases.append(case)
    
    return cases

# Load datasets
train_cases = load_cases('../data/synthetic_train.json')
test_cases = load_cases('../data/synthetic_test.json')

print(f"Loaded {len(train_cases)} training cases")
print(f"Loaded {len(test_cases)} test cases")

## 2. Initialize SRGL Systems

Initialize SRGL with different merging strategies for comparison.

In [ ]:
# Initialize systems
srgl_conservative = SRGL(merging_strategy='conservative')
srgl_average = SRGL(merging_strategy='average')
srgl_weighted = SRGL(merging_strategy='weighted')

print("Initialized SRGL systems:")
print("  1. Conservative merging (primary)")
print("  2. Average merging (baseline)")
print("  3. Weighted merging (baseline)")

## 3. Run Predictions on Test Set

In [ ]:
def run_predictions(srgl_system: SRGL, cases: List[PatientCase]) -> pd.DataFrame:
    """Run predictions and return results DataFrame."""
    results = []
    
    for case in cases:
        # Convert case to input format
        patient_data = {
            'patient_id': case.case_id,
            'age': case.patient_features['age'],
            'sex': case.patient_features['sex'],
            'symptoms': case.clinical_presentation.symptoms,
            'red_flags': case.clinical_presentation.red_flags,
            'risk_factors': case.clinical_presentation.risk_factors,
            'vital_signs': case.vital_signs,
            'lab_results': case.lab_results,
            'timestamp': case.timestamp
        }
        
        # Get prediction
        decision = srgl_system.predict(patient_data)
        
        results.append({
            'case_id': case.case_id,
            'ground_truth': case.ground_truth_tier.value,
            'predicted': decision.final_tier.value if decision.final_tier else 0,
            'confidence': decision.confidence,
            'is_abstention': decision.final_tier is None,
            'decision_time_ms': decision.processing_time_ms
        })
    
    return pd.DataFrame(results)

# Run predictions
print("Running predictions on test set...")
results_conservative = run_predictions(srgl_conservative, test_cases)
results_average = run_predictions(srgl_average, test_cases)
results_weighted = run_predictions(srgl_weighted, test_cases)

print("\nPredictions complete!")
print(f"Conservative - Abstention rate: {results_conservative['is_abstention'].mean()*100:.1f}%")
print(f"Average - Abstention rate: {results_average['is_abstention'].mean()*100:.1f}%")
print(f"Weighted - Abstention rate: {results_weighted['is_abstention'].mean()*100:.1f}%")

## 4. Calculate Safety Metrics

Calculate comprehensive safety metrics with confidence intervals.

In [ ]:
# Initialize metrics calculator for conservative strategy
y_true = results_conservative['ground_truth'].values
y_pred = results_conservative['predicted'].values

metrics = SafetyMetrics(y_true, y_pred)

# Calculate all metrics
sensitivity = metrics.sensitivity_critical(critical_threshold=3)
specificity = metrics.specificity_safe(safe_threshold=1)
fnr = metrics.false_negative_rate(critical_threshold=3)
unsafe_discharge = metrics.unsafe_discharge_rate(discharge_threshold=1)
abstention = metrics.abstention_rate()

print("="*60)
print("SRGL CONSERVATIVE MERGING - SAFETY METRICS")
print("="*60)
print(f"\nSensitivity (Critical Cases):")
print(f"  Value: {sensitivity['sensitivity']:.3f}")
print(f"  95% CI: [{sensitivity['ci_lower']:.3f}, {sensitivity['ci_upper']:.3f}]")
print(f"\nSpecificity (Safe Cases):")
print(f"  Value: {specificity['specificity']:.3f}")
print(f"  95% CI: [{specificity['ci_lower']:.3f}, {specificity['ci_upper']:.3f}]")
print(f"\nFalse Negative Rate:")
print(f"  Value: {fnr['fnr']:.3f}")
print(f"  95% CI: [{fnr['ci_lower']:.3f}, {fnr['ci_upper']:.3f}]")
print(f"\nUnsafe Discharge Rate:")
print(f"  Value: {unsafe_discharge['unsafe_discharge_rate']:.3f}")
print(f"  95% CI: [{unsafe_discharge['ci_lower']:.3f}, {unsafe_discharge['ci_upper']:.3f}]")
print(f"\nAbstention Rate:")
print(f"  Value: {abstention['abstention_rate']:.3f}")
print(f"  95% CI: [{abstention['ci_lower']:.3f}, {abstention['ci_upper']:.3f}]")
print("="*60)

## 5. Generate Summary Report

In [ ]:
# Generate comprehensive summary report
summary_report = metrics.summary_report()
print("\nCOMPREHENSIVE METRICS SUMMARY:")
print(summary_report.to_string())

# Save report
os.makedirs('../results', exist_ok=True)
summary_report.to_csv('../results/safety_metrics_summary.csv', index=False)
print("\nReport saved: results/safety_metrics_summary.csv")

## 6. Statistical Comparison: McNemar's Test

Compare SRGL conservative merging against baseline methods.

In [ ]:
# McNemar's test: Conservative vs Average
print("\nMcNemar's Test: Conservative vs Average Merging")
print("="*60)

result_avg = StatisticalTests.mcnemar_test(
    y_true=results_conservative['ground_truth'].values,
    y_pred1=results_conservative['predicted'].values,
    y_pred2=results_average['predicted'].values,
    critical_threshold=3
)

print(f"Statistic: {result_avg['statistic']:.3f}")
print(f"P-value: {result_avg['p_value']:.4f}")
print(f"Significant: {result_avg['significant']}")
print(f"Interpretation: {result_avg['interpretation']}")

# McNemar's test: Conservative vs Weighted
print("\n\nMcNemar's Test: Conservative vs Weighted Merging")
print("="*60)

result_weighted = StatisticalTests.mcnemar_test(
    y_true=results_conservative['ground_truth'].values,
    y_pred1=results_conservative['predicted'].values,
    y_pred2=results_weighted['predicted'].values,
    critical_threshold=3
)

print(f"Statistic: {result_weighted['statistic']:.3f}")
print(f"P-value: {result_weighted['p_value']:.4f}")
print(f"Significant: {result_weighted['significant']}")
print(f"Interpretation: {result_weighted['interpretation']}")

# Apply Bonferroni correction
p_values = [result_avg['p_value'], result_weighted['p_value']]
corrected = StatisticalTests.bonferroni_correction(p_values, alpha=0.05)

print("\n\nBonferroni Correction:")
print("="*60)
print(f"Adjusted alpha: {corrected['adjusted_alpha']:.4f}")
print(f"Significant comparisons: {corrected['significant']}")

## 7. Calibration Analysis

Evaluate uncertainty calibration using Expected Calibration Error (ECE).

In [ ]:
# Calculate ECE
# Convert predictions to probability format (confidence as probability)
# For binary classification: critical (R1-R3) vs non-critical (R4-R5)
y_true_binary = (results_conservative['ground_truth'] <= 3).astype(int)
y_pred_binary = (results_conservative['predicted'] <= 3).astype(int)
y_prob = results_conservative['confidence'].values

ece = CalibrationMetrics.expected_calibration_error(y_true_binary, y_prob, n_bins=10)

print("\nExpected Calibration Error (ECE):")
print("="*60)
print(f"ECE: {ece:.4f}")
print(f"\nInterpretation:")
if ece < 0.05:
    print("  Excellent calibration (ECE < 0.05)")
elif ece < 0.10:
    print("  Good calibration (0.05 ≤ ECE < 0.10)")
elif ece < 0.15:
    print("  Moderate calibration (0.10 ≤ ECE < 0.15)")
else:
    print("  Poor calibration (ECE ≥ 0.15)")
print("="*60)

In [ ]:
# Generate reliability diagram data
reliability_data = CalibrationMetrics.reliability_diagram_data(y_true_binary, y_prob, n_bins=10)

# Plot reliability diagram
fig, ax = plt.subplots(figsize=(8, 8))

# Plot perfect calibration line
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)

# Plot actual calibration
ax.plot(reliability_data['mean_predicted_prob'], 
        reliability_data['mean_true_prob'],
        'o-', linewidth=2, markersize=8, label='SRGL (Conservative)')

# Add error bars
for i in range(len(reliability_data['bin_centers'])):
    if reliability_data['counts'][i] > 0:
        ax.plot([reliability_data['mean_predicted_prob'][i]]*2,
               [reliability_data['mean_true_prob'][i] - 0.05,
                reliability_data['mean_true_prob'][i] + 0.05],
               'b-', linewidth=1, alpha=0.5)

ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Mean True Probability', fontsize=12)
ax.set_title(f'Reliability Diagram (ECE = {ece:.4f})', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('../figures/reliability_diagram.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: figures/reliability_diagram.png")

## 8. Confusion Matrix Analysis

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Filter out abstentions for confusion matrix
non_abstention_mask = ~results_conservative['is_abstention']
y_true_filtered = results_conservative[non_abstention_mask]['ground_truth'].values
y_pred_filtered = results_conservative[non_abstention_mask]['predicted'].values

# Calculate confusion matrix
cm = confusion_matrix(y_true_filtered, y_pred_filtered, labels=[1, 2, 3, 4, 5])

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['R1', 'R2', 'R3', 'R4', 'R5'],
            yticklabels=['R1', 'R2', 'R3', 'R4', 'R5'],
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Risk Tier', fontsize=12)
ax.set_ylabel('Ground Truth Risk Tier', fontsize=12)
ax.set_title('SRGL Confusion Matrix (Excluding Abstentions)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: figures/confusion_matrix.png")

# Print classification report
print("\n\nClassification Report:")
print("="*60)
print(classification_report(y_true_filtered, y_pred_filtered, 
                          target_names=['R1', 'R2', 'R3', 'R4', 'R5']))

## 9. Decision Time Analysis

In [ ]:
# Analyze decision time
decision_times = results_conservative['decision_time_ms'].values

print("\nDecision Time Statistics:")
print("="*60)
print(f"Mean: {decision_times.mean():.2f} ms")
print(f"Median: {np.median(decision_times):.2f} ms")
print(f"Std: {decision_times.std():.2f} ms")
print(f"Min: {decision_times.min():.2f} ms")
print(f"Max: {decision_times.max():.2f} ms")
print(f"95th percentile: {np.percentile(decision_times, 95):.2f} ms")
print("="*60)

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(decision_times, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(decision_times.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {decision_times.mean():.2f} ms')
axes[0].axvline(np.median(decision_times), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(decision_times):.2f} ms')
axes[0].set_xlabel('Decision Time (ms)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Decision Time Distribution', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot by risk tier
results_conservative.boxplot(column='decision_time_ms', by='ground_truth', ax=axes[1])
axes[1].set_xlabel('Ground Truth Risk Tier', fontsize=12)
axes[1].set_ylabel('Decision Time (ms)', fontsize=12)
axes[1].set_title('Decision Time by Risk Tier', fontsize=14, fontweight='bold')
axes[1].set_xticklabels(['R1', 'R2', 'R3', 'R4', 'R5'])
plt.suptitle('')

plt.tight_layout()
plt.savefig('../figures/decision_time_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: figures/decision_time_analysis.png")

## 10. Summary and Export Results

In [ ]:
# Create comprehensive results summary
results_summary = {
    'System': 'SRGL Conservative Merging',
    'Test Cases': len(test_cases),
    'Abstention Rate': f"{abstention['abstention_rate']:.3f} [{abstention['ci_lower']:.3f}, {abstention['ci_upper']:.3f}]",
    'Sensitivity': f"{sensitivity['sensitivity']:.3f} [{sensitivity['ci_lower']:.3f}, {sensitivity['ci_upper']:.3f}]",
    'Specificity': f"{specificity['specificity']:.3f} [{specificity['ci_lower']:.3f}, {specificity['ci_upper']:.3f}]",
    'FNR': f"{fnr['fnr']:.3f} [{fnr['ci_lower']:.3f}, {fnr['ci_upper']:.3f}]",
    'Unsafe Discharge Rate': f"{unsafe_discharge['unsafe_discharge_rate']:.3f} [{unsafe_discharge['ci_lower']:.3f}, {unsafe_discharge['ci_upper']:.3f}]",
    'ECE': f"{ece:.4f}",
    'Mean Decision Time (ms)': f"{decision_times.mean():.2f}",
    'McNemar vs Average': f"p={result_avg['p_value']:.4f} ({'sig' if result_avg['significant'] else 'ns'})",
    'McNemar vs Weighted': f"p={result_weighted['p_value']:.4f} ({'sig' if result_weighted['significant'] else 'ns'})"
}

# Convert to DataFrame
summary_df = pd.DataFrame([results_summary]).T
summary_df.columns = ['Value']

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(summary_df.to_string())
print("="*60)

# Save all results
summary_df.to_csv('../results/final_summary.csv')
results_conservative.to_csv('../results/predictions_conservative.csv', index=False)
results_average.to_csv('../results/predictions_average.csv', index=False)
results_weighted.to_csv('../results/predictions_weighted.csv', index=False)

print("\nAll results saved to results/ directory")
print("\nStatistical analysis complete! Proceed to notebook 03 for visualization.")